---
## Lab 12/05/2026 — Análise Gabor 1D: Par, Ímpar, Ondas P/T e Intervalos Clínicos

Esta seção implementa e analisa os seguintes itens solicitados:

| Item | Descrição |
|------|-----------|
| **Gabor PAR** | φ = 0 (cosseno), simétrico — detecta picos → pico R |
| **Gabor ÍMPAR** | φ = π/2 (seno), antissimétrico — detecta bordas → transições Q/S |
| **Energia combinada** | E = par² + ímpar² — localização independente de fase |
| **Detector Onda P** | Gabor de baixa frequência (f₀ = 6 Hz) na janela pré-R |
| **Detector Onda T** | Gabor de frequência muito baixa (f₀ = 4 Hz) na janela pós-R |
| **Intervalos** | RR, PR, QRS, ST, QT com propagação de erro |
| **Incerteza** | δt = √(δt_s² + δt_g²), onde δt_g = 1/(4π·f₀) |

### Fundamentação dos parâmetros

```
┌─────────┬──────────────────────────────┬────────┬───────────┐
│  Onda   │  Conteúdo espectral típico   │   f₀   │     σ     │
├─────────┼──────────────────────────────┼────────┼───────────┤
│  QRS    │ 10 – 25 Hz (pico ~15 Hz)     │ 15 Hz  │ 0,04 s   │
│  P      │  2 – 10 Hz (pico ~6 Hz)      │  6 Hz  │ 0,06 s   │
│  T      │  0,5 – 7 Hz (pico ~4 Hz)     │  4 Hz  │ 0,10 s   │
└─────────┴──────────────────────────────┴────────┴───────────┘
```

### Modelo de erro

$$
\delta t = \sqrt{\left(\frac{1}{f_s}\right)^2 + \left(\frac{1}{4\pi f_0}\right)^2}
\qquad
\delta I_{A\to B} = \sqrt{\delta t_A^2 + \delta t_B^2}
$$

Para `fs = 360 Hz` e `f₀ = 15 Hz` (QRS): **δt ≈ 5,4 ms**, portanto **δ(RR) ≈ 7,7 ms**.

In [ ]:
# ── Importa o módulo criado (copie ecg_gabor_analysis.py para src/ ou deixe na raiz) ──
import sys
sys.path.insert(0, "..")  # ajuste conforme a estrutura do projeto

from ecg_gabor_analysis import (
    make_gabor_kernel,
    gabor_energy,
    detect_R_peaks,
    detect_QS_waves,
    detect_P_waves,
    detect_T_waves,
    build_beat_features,
    beats_to_dataframe,
    beats_summary,
    plot_gabor_kernels,
    plot_all_waves,
    plot_interval_timeseries,
    plot_gabor_responses_ecg,
    QRS_F0, QRS_SIGMA, P_F0, P_SIGMA, T_F0, T_SIGMA,
)

print("Módulo carregado com sucesso.")

## 8. Kernels de Gabor: PAR, ÍMPAR e Espectro de Magnitude

A visualização a seguir compara os três kernels (QRS, P, T) nas suas versões **par** (cosseno, φ=0) e **ímpar** (seno, φ=π/2), além do seu espectro de magnitude, confirmando a banda passante sintonizada para cada onda.

**Por que usar os dois?**

* O Gabor **par** máxima a resposta quando o evento tem simetria de pico (caso do R e do pico T).
* O Gabor **ímpar** máxima quando o evento é uma transição ou borda (subida/descida do QRS — útil para Q e S).
* A **energia combinada** E = par² + ímpar² elimina a dependência de fase e provê um envelope suave.

In [ ]:
plot_gabor_kernels(fs=fs)

## 9. Respostas Gabor sobre o Sinal de ECG

Neste painel vemos as respostas **par**, **ímpar** e a **energia** de cada filtro aplicados ao sinal bruto dos primeiros 10 s. 

* Observe como a energia do filtro QRS forma picos bem localizados coincidindo com os batimentos anotados.
* A energia do filtro P tem picos menores posicionados ~150 ms antes de cada R.
* A energia do filtro T apresenta picos suaves ~200–350 ms após cada R.

In [ ]:
plot_gabor_responses_ecg(
    t_sig=t,
    x=x_raw,
    fs=fs,
    ann_samples=ann_in_window,
)

## 10. Pipeline Completo de Detecção de Ondas

O pipeline abaixo executa em cascata:

1. **Detecção do R** — energia Gabor QRS, refinamento dentro de ±80 ms da anotação.
2. **Detecção de Q e S** — mínimos locais do sinal dentro de ±60 ms ao redor do R.
3. **Detecção da P** — energia Gabor P na janela \[R − 280 ms, R − 80 ms\].
4. **Detecção da T** — energia Gabor T na janela \[R + 120 ms, R + 500 ms\].

> **Nota:** O sinal bruto (`x_raw`) é usado deliberadamente para evitar que o ringing do filtro FIR afete a posição dos mínimos Q/S.

In [ ]:
# ── Executa o pipeline completo ──────────────────────────────────────────────
beats = build_beat_features(
    x=x_raw,
    fs=fs,
    ann_samples=ann_in_window,
    qrs_f0=QRS_F0,
    qrs_sigma=QRS_SIGMA,
    p_f0=P_F0,
    p_sigma=P_SIGMA,
    t_f0=T_F0,
    t_sigma=T_SIGMA,
)

print(f"Batimentos detectados: {len(beats)}")

# Mostra as ondas sobre o sinal
plot_all_waves(
    t_sig=t,
    x=x_raw,
    beats=beats,
    fs=fs,
    title=f"MIT BIH {RECORD_NAME}: Detecção PQRST com Barras de Erro (±1σ)",
)

## 11. Métricas por Batimento e Margens de Erro

A tabela abaixo apresenta, para cada batimento detectado na janela de 10 s:

| Coluna | Descrição |
|--------|----------|
| `R (s)` | Tempo do pico R refinado pelo Gabor |
| `P/Q/S/T (s)` | Tempos das ondas detectadas |
| `RR (ms) ± err` | Intervalo entre picos R consecutivos |
| `PR (ms) ± err` | P-pico → R-pico (proxy do intervalo PQ) |
| `QRS (ms) ± err`| Q-mínimo → S-mínimo (duração do QRS) |
| `ST (ms) ± err` | S-mínimo → T-pico |
| `QT (ms) ± err` | Q-mínimo → T-pico |
| `Flags` | Intervalos fora das faixas clínicas normais |

As colunas `±` expressam **1σ propagado** da incerteza δt de cada ponto detector.

**Tabela de incertezas esperadas (fs = 360 Hz):**

| Onda | f₀ (Hz) | δt_s (ms) | δt_g (ms) | δt (ms) |
|------|---------|-----------|-----------|----------|
| R    | 15      | 2,78      | 5,31      | **6,0**  |
| P    |  6      | 2,78      | 13,3      | **13,6** |
| T    |  4      | 2,78      | 19,9      | **20,1** |
| Q, S | —      | 2,78      | 3,93 (2 amostras) | **4,8** |

Portanto os intervalos têm incerteza típica de:
- **±RR ≈ 8–9 ms** (R→R)
- **±PR ≈ 15 ms** (P→R)
- **±QRS ≈ 7 ms** (Q→S)
- **±ST ≈ 21 ms** (S→T)
- **±QT ≈ 21 ms** (Q→T)

In [ ]:
df_beats = beats_to_dataframe(beats)

# Formatação: destaca linhas com flags
def highlight_flags(row):
    if row["Flags"]:
        return ["background-color: #fff3cd"] * len(row)
    return [""] * len(row)

display(
    df_beats.style
        .apply(highlight_flags, axis=1)
        .format(precision=1, na_rep="—")
        .set_caption("Métricas por Batimento — MIT BIH 100 (primeiros 10 s)")
)

## 12. Sumário Estatístico dos Intervalos

In [ ]:
summary = beats_summary(df_beats)

display(
    summary.style
        .set_caption("Sumário Estatístico dos Intervalos (ms)")
        .format(precision=1)
        .background_gradient(subset=["Média (ms)"], cmap="RdYlGn", axis=0)
)

## 13. Série Temporal dos Intervalos com Bandas de Erro

O painel abaixo mostra a **evolução temporal** de cada intervalo batimento a batimento. As faixas sombreadas representam:

* **Banda azul-clara ± 1σ** — incerteza de medição propagada.
* **Banda cinza** — faixa de normalidade clínica de referência.

Esta representação é especialmente importante para o **treinamento futuro do modelo de predição de anomalias**: a comparação entre a variabilidade fisiológica esperada e a incerteza instrumental permite definir limiares de decisão realistas, evitando falsos positivos gerados por erros de medição.

In [ ]:
plot_interval_timeseries(
    beats,
    title=f"MIT BIH {RECORD_NAME}: Intervalos ECG com Bandas de Erro e Referência Clínica",
)

## 14. Exportação para CSV (entrada do modelo futuro)

O DataFrame de métricas é exportado em formato tabulado para ser consumido pelo pipeline de treinamento do modelo. As colunas de erro (`±`) devem ser mantidas pois podem ser usadas como **pesos de confiança** (sample weights) no treinamento supervisionado.

In [ ]:
# Adiciona metadados relevantes para o modelo
df_export = df_beats.copy()
df_export.insert(0, "record",   RECORD_NAME)
df_export.insert(1, "fs_hz",    int(fs))
df_export.insert(2, "channel",  "MLII")

# Frequência cardíaca instantânea (bpm) a partir do RR
df_export["HR_bpm"] = (60_000 / df_export["RR (ms)"]).round(1)

# QTc corrigido pela FC — fórmula de Bazett: QTc = QT / √(RR/1000)
df_export["QTc_Bazett_ms"] = (
    df_export["QT (ms)"] / np.sqrt(df_export["RR (ms)"] / 1000.0)
).round(1)

out_path = f"../data/processed/features_{RECORD_NAME}_10s.csv"
df_export.to_csv(out_path, index=False)
print(f"Exportado: {out_path}")
display(df_export.head())

## 15. Próximos Passos (Fase 2)

Com o pipeline de extração de features operacional, as etapas seguintes para o modelo de predição de anomalias são:

1. **Ampliar para todos os registros MIT-BIH** (48 registros, 30 min cada) gerando o dataset completo de features.
2. **Engenharia de features adicionais**:
   - HRV (Heart Rate Variability): SDNN, RMSSD, pNN50 sobre séries RR.
   - QTc corrigido (Fridericia: QTc = QT / RR^(1/3)).
   - Morfologia do template QRS (cross-correlation com template de referência).
3. **Balanceamento de classes**: os registros MIT-BIH são desbalanceados (ritmo normal >> arritmias). Avaliar SMOTE ou class_weight.
4. **Modelo baseline**: Random Forest ou XGBoost usando as features tabeladas.
5. **Modelo avançado**: CNN/LSTM sobre os segmentos brutos PQRST normalizados.
6. **Calibração de incerteza**: usar as colunas `±` como pesos de confiança (`sample_weight`) no treinamento.

**Obs. sobre as margens de erro no contexto do modelo:**
> A incerteza δt_g do Gabor cresce com 1/f₀. Para a onda T (f₀ = 4 Hz), δt ≈ 20 ms, o que representa ~5% do intervalo QT médio. Este nível de ruído de medição deve ser considerado ao definir limiares de alerta clínico — por exemplo, um QT "longo" só deve ser sinalizado se QT > 450 + 2·δ(QT) ms, evitando falsos positivos instrumentais.